In [6]:
import torch
from torchtext.datasets import IMDB
from torch.utils.data import DataLoader
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import torch.nn as nn

train_iter = IMDB(split='train')

label, text = next(iter(train_iter))
print(f"Label: {label}")  # 1 = positive, 2 = negative
print(f"Text: {text[:200]}") 

Label: 1
Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev


In [5]:
tokenizer= get_tokenizer('basic_english')

def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)
        
vocab= build_vocab_from_iterator(yield_tokens(train_iter), specials=['<unk>', '<pad>'])
vocab.set_default_index(vocab['<unk>'])

print(f"Vocabulary size: {len(vocab)}")

def text_pipeline(text):
    return vocab(tokenizer(text))


def label_pipeline(label):
    return 1 if label == 2 else 0  

sample_text = "This movie was shit ass pathetic!"
print(f"Text: {sample_text}")
print(f"Tokens: {tokenizer(sample_text)}")
print(f"IDs: {text_pipeline(sample_text)}")

Vocabulary size: 68812
Text: This movie was shit ass pathetic!
Tokens: ['this', 'movie', 'was', 'shit', 'ass', 'pathetic', '!']
IDs: [14, 18, 17, 22531, 2000, 723, 35]


In [7]:
class TextRNN(nn.Module):
    def __init__(self,vocab_size, embed_dim, hidden_dim, output_dim):
        super().__init__()

        self.embedding= nn.Embedding(vocab_size, embed_dim)
        self.rnn= nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    
    def forward(self, text):
        embedded= self.embedding(text)
        out, hidden= self.rnn(embedded)
        out= self.fc(out[:,-1,:])

        return out 
    
vocab_size = len(vocab)
model = TextRNN(vocab_size=vocab_size, embed_dim=100, hidden_dim=256, output_dim=1)
print(model)

TextRNN(
  (embedding): Embedding(68812, 100)
  (rnn): RNN(100, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
)
